# EoH Full Compare (Shared Initial Population)

Runs baseline and routed with the same parameters and the same initial population per seed.

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

NB_DIR = Path.cwd().resolve()
PROJECT_ROOT = NB_DIR.parent if (NB_DIR / 'run_compare_baseline_routed.py').exists() else NB_DIR
COMPARE_ROOT = PROJECT_ROOT / 'compare_runs'

print('NOTEBOOK_DIR:', NB_DIR)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('COMPARE_ROOT:', COMPARE_ROOT)


## Settings
- Same params for baseline/routed
- `EOH_SHARE_INITIAL_POP=1` ensures both modes start from exactly the same initial population per seed
- Multi-seed default: `2024,2025,2026`

In [ ]:
# ENSIA HPC defaults
os.environ['ENSIA_VLLM_BASE'] = 'http://vllm-nodeport.vllm-ns.svc.cluster.local:8000/v1'
os.environ['ENSIA_VLLM_API_KEY'] = 'my-key-ensia-2022-1030'
os.environ['ENSIA_VLLM_MODEL'] = 'QuantTrio/Qwen3-VL-235B-A22B-Instruct-AWQ'

# Compare settings
os.environ['EOH_POP_SIZE'] = '8'
os.environ['EOH_N_GENERATIONS'] = '10'
os.environ['EOH_EVAL_INSTANCES_PER_GEN'] = '256'
os.environ['EOH_HOLDOUT_INSTANCES'] = '64'
os.environ['EOH_HOLDOUT_EVAL_INTERVAL'] = '1'
os.environ['EOH_ROUTE_IMPROVEMENT_EPS'] = '1e-12'
os.environ['EOH_ROUTE_WARMUP_GENS'] = '2'
os.environ['EOH_ROUTE_E1_COOLDOWN'] = '3'
os.environ['EOH_ROUTE_E2_RECENT_K'] = '3'
os.environ['EOH_ROUTE_USE_PROBABILISTIC'] = '1'
os.environ['EOH_N_PROC'] = '1'
os.environ['EOH_DISABLE_NUMBA'] = '1'
os.environ['EOH_LOG_LLM_IO'] = '1'
os.environ['EOH_SHARE_INITIAL_POP'] = '1'
os.environ['EOH_COMPARE_OUT'] = str(COMPARE_ROOT)
os.environ['EOH_COMPARE_SEEDS'] = '2024,2025,2026'

print({k: os.environ.get(k) for k in [
    'EOH_POP_SIZE',
    'EOH_N_GENERATIONS',
    'EOH_EVAL_INSTANCES_PER_GEN',
    'EOH_HOLDOUT_INSTANCES',
    'EOH_HOLDOUT_EVAL_INTERVAL',
    'EOH_ROUTE_IMPROVEMENT_EPS',
    'EOH_ROUTE_WARMUP_GENS',
    'EOH_ROUTE_E1_COOLDOWN',
    'EOH_ROUTE_E2_RECENT_K',
    'EOH_ROUTE_USE_PROBABILISTIC',
    'EOH_SHARE_INITIAL_POP',
    'EOH_COMPARE_SEEDS',
]})


In [ ]:
# Run baseline+routed compare
status_log = COMPARE_ROOT / 'runner_status.jsonl'
print('Runner status log:', status_log)
cmd = [sys.executable, '-u', str(PROJECT_ROOT / 'notebooks' / 'run_compare_baseline_routed.py')]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)


In [ ]:
# Verify baseline/routed share the same initial objectives per seed
seeds = [int(s.strip()) for s in os.environ.get('EOH_COMPARE_SEEDS', '2024,2025,2026').split(',') if s.strip()]
for seed in seeds:
    b0 = COMPARE_ROOT / f'seed_{seed}' / 'baseline' / 'results' / 'pops' / 'population_generation_0.json'
    r0 = COMPARE_ROOT / f'seed_{seed}' / 'routed' / 'results' / 'pops' / 'population_generation_0.json'
    ok = False
    if b0.exists() and r0.exists():
        b = json.loads(b0.read_text(encoding='utf-8'))
        r = json.loads(r0.read_text(encoding='utf-8'))
        b_obj = [x.get('objective') for x in b if isinstance(x, dict)]
        r_obj = [x.get('objective') for x in r if isinstance(x, dict)]
        ok = (b_obj == r_obj)
    print(f'seed={seed} same_initial_objectives={ok} baseline_pop0={b0.exists()} routed_pop0={r0.exists()}')


In [ ]:
# Generate aggregate plots
cmd = [
    sys.executable,
    '-u',
    str(PROJECT_ROOT / 'notebooks' / 'plot_run_log.py'),
    '--root',
    str(COMPARE_ROOT),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)


In [ ]:
from IPython.display import Image, display

plots_dir = COMPARE_ROOT / 'plots'
plot_files = [
    'fitness_vs_gen.png',
    'operator_over_time.png',
    'invalid_and_diversity.png',
    'operator_effect_by_mode.png',
    'diagnosis_effect_routed.png',
]
print('plots_dir:', plots_dir)
for name in plot_files:
    p = plots_dir / name
    print(name, 'exists=', p.exists())
    if p.exists():
        display(Image(filename=str(p)))


In [ ]:
# Tail runner status log
status_log = COMPARE_ROOT / 'runner_status.jsonl'
if status_log.exists():
    lines = status_log.read_text(encoding='utf-8').splitlines()
    print('\n'.join(lines[-30:]))
else:
    print('No runner_status.jsonl yet')
